In [0]:
# NB_lab_test_trends — Gold Layer: Lab Test Trends
# STTM: One row per test type (test_name) per calendar month
# Load Strategy: FULL — CREATE OR REPLACE TABLE ... AS SELECT
# Sources: silver.lab_results (driving), silver.dim_date (optional join)
# Window: LAG() partitioned by test_name, ordered by year, month

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
import uuid
from datetime import datetime

# --- Configuration ---
CATALOG = "caresync_databricks_workspace"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"
TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.lab_test_trends"

RUN_ID = str(uuid.uuid4())
RUN_START = datetime.utcnow()

print("=" * 60)
print("NB_lab_test_trends — Gold Layer Pipeline")
print("=" * 60)
print(f"  Run ID             : {RUN_ID}")
print(f"  Run Start (UTC)    : {RUN_START}")
print(f"  Target Table       : {TARGET_TABLE}")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 1: Read Source Tables")
print("=" * 60)

lab_results_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.lab_results")
dim_date_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.dim_date")

lab_results_count = lab_results_df.count()
dim_date_count = dim_date_df.count()

print(f"  lab_results   : {lab_results_count:,} rows")
print(f"  dim_date      : {dim_date_count:,} rows")
print("=" * 60)

In [0]:
# Driving table: lab_results grouped by (test_name, year(test_date), month(test_date))
# This establishes the grain: one row per test type per calendar month with activity.

print("=" * 60)
print("STEP 2: Aggregate Lab Results by Test Name + Year + Month")
print("=" * 60)

monthly_agg_df = (
    lab_results_df
    .withColumn("year", F.year("test_date"))
    .withColumn("month", F.month("test_date"))
    .groupBy("test_name", "year", "month")
    .agg(
        F.count("*").alias("total_tests"),
        F.count(F.when(F.col("result_status") == "Abnormal", 1)).alias("abnormal_test_count"),
        F.count(F.when(F.col("result_status") == "Critical", 1)).alias("critical_test_count")
    )
)

agg_count = monthly_agg_df.count()
print(f"  Aggregated rows (test_name x year x month) : {agg_count:,}")
print("=" * 60)

In [0]:
# Join dim_date on the first day of the month to fetch month_name and quarter metadata.
# STTM referenced quarter_name from calendar; dim_date has quarter (int), so derive quarter_name as CONCAT('Q', quarter).
# Compute rates with safe division and LAG over available observations (not gap-filled calendar months).

print("=" * 60)
print("STEP 3: Join dim_date and Compute Derived Columns")
print("=" * 60)

# Derive month start date and date_key for optional join to dim_date
with_month_start_df = (
    monthly_agg_df
    .withColumn(
        "month_start_date",
        F.make_date(F.col("year"), F.col("month"), F.lit(1))
    )
    .withColumn(
        "_date_key",
        F.date_format(F.col("month_start_date"), "yyyyMMdd").cast("int")
    )
)

joined_df = (
    with_month_start_df.alias("m")
    .join(
        dim_date_df.alias("cal"),
        F.col("m._date_key") == F.col("cal.date_key"),
        "left"
    )
)

with_rates_df = (
    joined_df
    .withColumn(
        "abnormal_rate_pct",
        F.when(
            F.col("m.total_tests") == 0, F.lit(None)
        ).otherwise(
            F.round(F.col("m.abnormal_test_count") / F.col("m.total_tests") * 100, 2)
        ).cast("decimal(5,2)")
    )
    .withColumn(
        "critical_rate_pct",
        F.when(
            F.col("m.total_tests") == 0, F.lit(None)
        ).otherwise(
            F.round(F.col("m.critical_test_count") / F.col("m.total_tests") * 100, 2)
        ).cast("decimal(5,2)")
    )
    .withColumn(
        "quarter_name",
        F.concat(F.lit("Q"), F.col("cal.quarter").cast("string"))
    )
)

# Window for LAG over available observations per test_name
w = Window.partitionBy("m.test_name").orderBy("m.year", "m.month")

with_lag_df = (
    with_rates_df
    .withColumn(
        "prev_month_abnormal_rate_pct",
        F.lag("abnormal_rate_pct").over(w).cast("decimal(5,2)")
    )
    .withColumn(
        "abnormal_rate_change_pct",
        F.when(
            F.col("prev_month_abnormal_rate_pct").isNull(),
            F.lit(None)
        ).otherwise(
            (F.col("abnormal_rate_pct") - F.col("prev_month_abnormal_rate_pct"))
        ).cast("decimal(6,2)")
    )
)

lab_test_trends_df = with_lag_df.select(
    F.col("m.test_name").cast("string").alias("test_name"),
    F.col("m.year").cast("int").alias("year"),
    F.col("m.month").cast("int").alias("month"),
    F.col("cal.month_name").cast("string").alias("month_name"),
    F.col("quarter_name").cast("string").alias("quarter_name"),
    F.col("m.total_tests").cast("int").alias("total_tests"),
    F.col("m.abnormal_test_count").cast("int").alias("abnormal_test_count"),
    F.col("m.critical_test_count").cast("int").alias("critical_test_count"),
    F.col("abnormal_rate_pct"),
    F.col("critical_rate_pct"),
    F.col("prev_month_abnormal_rate_pct"),
    F.col("abnormal_rate_change_pct")
)

print(f"  Transformation built successfully.")
print(f"  Output columns: {lab_test_trends_df.columns}")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 4: Pre-write Validation")
print("=" * 60)

lab_test_trends_df.createOrReplaceTempView("lab_test_trends_staging")
staging_count = spark.sql("SELECT COUNT(*) AS cnt FROM lab_test_trends_staging").collect()[0]["cnt"]

print(f"  Staging row count: {staging_count:,}")

# Validation 1: Grain uniqueness (test_name + year + month)
grain_dupes = spark.sql("""
    SELECT test_name, year, month, COUNT(*) AS cnt
    FROM lab_test_trends_staging
    GROUP BY test_name, year, month
    HAVING cnt > 1
""").count()
assert grain_dupes == 0, f"FAIL: {grain_dupes} duplicate (test_name, year, month) found."
print(f"  [PASS] Grain unique: (test_name, year, month) — 0 duplicates")

# Validation 2: No null grain keys
null_grain = spark.sql("""
    SELECT COUNT(*) AS cnt FROM lab_test_trends_staging 
    WHERE test_name IS NULL OR year IS NULL OR month IS NULL
""").collect()[0]["cnt"]
assert null_grain == 0, f"FAIL: {null_grain} rows with null grain keys."
print(f"  [PASS] No null grain keys")

# Validation 3: total_tests > 0
zero_tests = spark.sql("SELECT COUNT(*) AS cnt FROM lab_test_trends_staging WHERE total_tests <= 0").collect()[0]["cnt"]
assert zero_tests == 0, f"FAIL: {zero_tests} rows with total_tests <= 0."
print(f"  [PASS] All total_tests >= 1")

# Validation 4: Rates in valid range [0, 100]
bad_rates = spark.sql("""
    SELECT COUNT(*) AS cnt FROM lab_test_trends_staging 
    WHERE (abnormal_rate_pct < 0 OR abnormal_rate_pct > 100)
       OR (critical_rate_pct < 0 OR critical_rate_pct > 100)
""").collect()[0]["cnt"]
assert bad_rates == 0, f"FAIL: {bad_rates} rows with rates outside [0,100]."
print(f"  [PASS] All rates in [0, 100] range")

# Validation 5: Source row count reconciliation
# Sum of total_tests across all groups should equal lab_results_count
total_tests_sum = spark.sql("SELECT SUM(total_tests) AS s FROM lab_test_trends_staging").collect()[0]["s"]
assert total_tests_sum == lab_results_count, (
    f"FAIL: SUM(total_tests) {total_tests_sum} != lab_results count {lab_results_count}"
)
print(f"  [PASS] SUM(total_tests) = {total_tests_sum:,} matches source lab_results ({lab_results_count:,})")

# Validation 6: Schema check
expected_cols = [
    "test_name", "year", "month", "month_name", "quarter_name",
    "total_tests", "abnormal_test_count", "critical_test_count",
    "abnormal_rate_pct", "critical_rate_pct", "prev_month_abnormal_rate_pct",
    "abnormal_rate_change_pct"
]
actual_cols = [f.name for f in spark.sql("SELECT * FROM lab_test_trends_staging LIMIT 0").schema.fields]
assert actual_cols == expected_cols, f"FAIL: Column mismatch.\nExpected: {expected_cols}\nActual:   {actual_cols}"
print(f"  [PASS] Schema matches STTM specification ({len(expected_cols)} columns)")

print("-" * 60)
print(f"  All pre-write validations PASSED.")
print("=" * 60)

In [0]:
print("=" * 60)
print(f"STEP 5: Write to Gold — {TARGET_TABLE}")
print("=" * 60)
print(f"  Strategy: CREATE OR REPLACE TABLE ... AS SELECT")

spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE} AS
    SELECT * FROM lab_test_trends_staging
""")

print(f"  ✅ Table created/replaced successfully.")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 6: Post-write Verification")
print("=" * 60)

gold_df = spark.table(TARGET_TABLE)
gold_count = gold_df.count()

print(f"  Gold table row count   : {gold_count:,}")
print(f"  Expected (from staging): {staging_count:,}")
assert gold_count == staging_count, f"FAIL: Gold count {gold_count} != staging count {staging_count}"
print(f"  [PASS] Row counts match.")

print("-" * 60)
print("  Schema:")
gold_df.printSchema()
print("-" * 60)
print("  Sample rows (5):")
gold_df.show(5, truncate=False)
print("=" * 60)

In [0]:
from datetime import datetime
import json

RUN_END = datetime.utcnow()
duration_sec = (RUN_END - RUN_START).total_seconds()

exit_payload = {
    "status": "SUCCESS",
    "run_id": RUN_ID,
    "target_table": TARGET_TABLE,
    "rows_written": gold_count,
    "source_lab_results_count": lab_results_count,
    "unique_test_month_combos": staging_count,
    "run_start_utc": str(RUN_START),
    "run_end_utc": str(RUN_END),
    "duration_seconds": round(duration_sec, 2)
}

print("=" * 60)
print("STEP 7: Notebook Exit")
print("=" * 60)
for k, v in exit_payload.items():
    print(f"  {k:30s}: {v}")
print("=" * 60)

In [0]:
record_count = gold_count

print("=" * 60)
print("STEP 8: Simple Record Count Exit")
print("=" * 60)
print(f"  Records processed  : {record_count:,}")
print(f"  Returning value    : {record_count}")
print("=" * 60)

dbutils.notebook.exit(str(record_count))